Purpose: Run pathway enrichment tests (Fisher's exact) on HybridExpress results.<br>
Author: Anna Pardo<br>
Date initiated: Aug. 4, 2026

In [1]:
# import modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statistics
import scipy.stats as stats
import numpy as np
from statsmodels.stats.multitest import fdrcorrection
from venn import venn
from matplotlib.patches import Patch
import json

In [2]:
# load pathway annotation
pathann = pd.read_csv("/home/leviathan22/yucca-genomics/photresp_N_citrate_genes_Yucca.csv",sep=",",header="infer")
pathann = pathann[~pathann["Pathway"].isin(["CAM-dark","CAM-light"])]
pathann.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [3]:
camannot = pd.read_csv("/home/leviathan22/yucca-genomics/camgenes_Ya_Yf_orthology_synteny.txt",sep="\t",header="infer")

In [4]:
clockann = pd.read_csv("/home/leviathan22/yucca-genomics/circadian_light_genes_by_orthology_Yucca.csv",sep=",",header="infer")

In [5]:
camannot = camannot[["Pathway","gene_abbr","GeneID","subgenome","gene_abbr_unique"]].rename(columns={
    "gene_abbr":"gene_family","gene_abbr_unique":"gene_name_unique"
})

In [6]:
clockann["Pathway"] = "Clock"
clockann = clockann[["Pathway","gene_name","GeneID","subgenome","gene_name_unique"]].rename(columns={"gene_name":"gene_family"})
clockann.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,Clock,TOC1,Yucal.11G006100.v2.1,Ya,Ya_TOC1_1
1,Clock,TOC1,Yucal.15G098200.v2.1,Ya,Ya_TOC1_2
2,Clock,TOC1,Yucal.16G106200.v2.1,Ya,Ya_TOC1_3
3,Clock,TOC1,YufilH1057026m.g,Yf,Yf_TOC1_1
4,Clock,TOC1,YufilH1057027m.g,Yf,Yf_TOC1_2


In [7]:
allpath = pd.concat([pathann,camannot,clockann])

In [8]:
# load HybridExpress results
res = pd.read_csv("./hybexp_top30pct_partitiongenes_bygttreat.txt",sep="\t",header="infer")
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat
0,recip_syn1000,1,ADD,5.957692,-0.447832,18_D
1,recip_syn10008,1,ADD,0.877055,-1.115803,18_D
2,recip_syn10040,1,ADD,3.715160,-1.074124,18_D
3,recip_syn10057,1,ADD,1.211627,-0.911991,18_D
4,recip_syn10061,1,ADD,3.105884,-1.503516,18_D


In [9]:
# split genotype & treatment info
gtt = res["genotype_treat"].str.split("_",expand=True)
gtt.rename(columns={0:"genotype",1:"treat"},inplace=True)
res = pd.concat([res,gtt],axis=1)
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat
0,recip_syn1000,1,ADD,5.957692,-0.447832,18_D,18,D
1,recip_syn10008,1,ADD,0.877055,-1.115803,18_D,18,D
2,recip_syn10040,1,ADD,3.715160,-1.074124,18_D,18,D
3,recip_syn10057,1,ADD,1.211627,-0.911991,18_D,18,D
4,recip_syn10061,1,ADD,3.105884,-1.503516,18_D,18,D


In [10]:
physcat = json.load(open("/home/leviathan22/yucca-genomics/physiology/physiological_categories_from_TA.json"))

In [11]:
res["phys"] = res["genotype"].map(physcat)
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat,phys
0,recip_syn1000,1,ADD,5.957692,-0.447832,18_D,18,D,C3+CAM
1,recip_syn10008,1,ADD,0.877055,-1.115803,18_D,18,D,C3+CAM
2,recip_syn10040,1,ADD,3.715160,-1.074124,18_D,18,D,C3+CAM
3,recip_syn10057,1,ADD,1.211627,-0.911991,18_D,18,D,C3+CAM
4,recip_syn10061,1,ADD,3.105884,-1.503516,18_D,18,D,C3+CAM


In [12]:
# Fisher table setup: isPathway Y/N, isClass Y/N
## run separately for each treatment

# for total genes: load reciprocal syntelogs
syn = pd.read_csv("/home/leviathan22/Yucca_genomics/yucca_synteny/reciprocal_syntelogs_sameinYaYf.txt",
                 sep="\t",header="infer")
syn.head()

,Yf_GeneID,Ya_GeneID,syntelogID
0,YufilH1000002m.g,Yucal.01G000100.v2.1,recip_syn1
1,YufilH1000007m.g,Yucal.01G000200.v2.1,recip_syn2
2,YufilH1000011m.g,Yucal.01G000400.v2.1,recip_syn3
3,YufilH1000013m.g,Yucal.01G000500.v2.1,recip_syn4
4,YufilH1000018m.g,Yucal.01G000600.v2.1,recip_syn5


In [13]:
pathdict = {}
for i in allpath["Pathway"].unique():
    if i.startswith("CAM"):
        pathdict[i] = "CAM"
    else:
        pathdict[i] = i

In [14]:
allpath["Pathway"] = allpath["Pathway"].map(pathdict)

In [15]:
synlong = pd.read_csv("/home/leviathan22/Yucca_genomics/yucca_synteny/reciprocal_syntelogs_sameinYaYf_long.txt",
                     sep="\t",header="infer")

In [16]:
allpath = allpath.merge(synlong)
allpath.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique,syntelogID
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1,recip_syn2455
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2,recip_syn28929
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1,recip_syn2455
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2,recip_syn13442
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3,recip_syn28929


In [17]:
res.rename(columns={"Gene":"syntelogID"},inplace=True)
res.head()

,syntelogID,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat,phys
0,recip_syn1000,1,ADD,5.957692,-0.447832,18_D,18,D,C3+CAM
1,recip_syn10008,1,ADD,0.877055,-1.115803,18_D,18,D,C3+CAM
2,recip_syn10040,1,ADD,3.715160,-1.074124,18_D,18,D,C3+CAM
3,recip_syn10057,1,ADD,1.211627,-0.911991,18_D,18,D,C3+CAM
4,recip_syn10061,1,ADD,3.105884,-1.503516,18_D,18,D,C3+CAM


In [18]:
syn.head()

,Yf_GeneID,Ya_GeneID,syntelogID
0,YufilH1000002m.g,Yucal.01G000100.v2.1,recip_syn1
1,YufilH1000007m.g,Yucal.01G000200.v2.1,recip_syn2
2,YufilH1000011m.g,Yucal.01G000400.v2.1,recip_syn3
3,YufilH1000013m.g,Yucal.01G000500.v2.1,recip_syn4
4,YufilH1000018m.g,Yucal.01G000600.v2.1,recip_syn5


In [30]:
## updated code
# Precompute constants
pathways = allpath["Pathway"].unique()
classes = res["Class"].unique()

# Base syntelog IDs
synids = syn["syntelogID"]


def make_membership_table(ptype, treat):

    df = pd.DataFrame({"syntelogID": synids})

    # -----------------------
    # Class membership
    # -----------------------

    r = res

    if ptype != "all":
        r = r[r["phys"] == ptype]

    if treat != "all":
        r = r[r["treat"] == treat]

    for cls in classes:
        ids = r.loc[r["Class"] == cls, "syntelogID"]
        df["is" + cls] = df["syntelogID"].isin(ids).to_numpy()

    # -----------------------
    # Pathway membership
    # -----------------------

    for path in pathways:
        ids = allpath.loc[
            allpath["Pathway"] == path,
            "syntelogID"
        ]
        df["is" + path] = df["syntelogID"].isin(ids).to_numpy()

    return df


def run_fisher(path, cls, df):

    p = df["is" + path].to_numpy()
    c = df["is" + cls].to_numpy()

    a = np.sum(p & c)
    b = np.sum(p & ~c)
    c2 = np.sum(~p & c)
    d = np.sum(~p & ~c)

    odds_ratio, p_value = stats.fisher_exact(
        [[a, b],
         [c2, d]]
    )

    return odds_ratio, p_value


# Build membership tables
mtables = {}

for treat in treats:
    for ptype in ptypes:
        mtables[ptype + "_" + treat] = make_membership_table(
            ptype, treat
        )


def run_enrich_all(treat, ptype):

    mtbl = mtables[ptype + "_" + treat]

    results = []

    for path in pathways:
        for cls in classes:

            odds_ratio, p_value = run_fisher(
                path, cls, mtbl
            )

            results.append({
                "Pathway": path,
                "Class": cls,
                "treatment": treat,
                "physiotype": ptype,
                "p-value": p_value,
                "odds_ratio": odds_ratio
            })

    return pd.DataFrame(results)




In [31]:
dfl = [
    run_enrich_all(treat, ptype)
    for treat in treats
    for ptype in ptypes
]

enrich_results = pd.concat(dfl, ignore_index=True)

In [32]:
enrich_results.head()

,Pathway,Class,treatment,physiotype,p-value,odds_ratio
0,PhotResp,ADD,all,C3+CAM,0.268864,1.544015
1,PhotResp,ELD_P2,all,C3+CAM,0.025796,2.660047
2,PhotResp,DOWN,all,C3+CAM,0.191784,0.563678
3,PhotResp,ELD_P1,all,C3+CAM,0.025862,0.408382
4,PhotResp,UP,all,C3+CAM,0.014598,2.532372


In [33]:
# add FDR p-values
enrich_results["FDR_p"] = fdrcorrection(enrich_results["p-value"])[1]
enrich_results.head()

,Pathway,Class,treatment,physiotype,p-value,odds_ratio,FDR_p
0,PhotResp,ADD,all,C3+CAM,0.268864,1.544015,0.556631
1,PhotResp,ELD_P2,all,C3+CAM,0.025796,2.660047,0.158337
2,PhotResp,DOWN,all,C3+CAM,0.191784,0.563678,0.491754
3,PhotResp,ELD_P1,all,C3+CAM,0.025862,0.408382,0.158337
4,PhotResp,UP,all,C3+CAM,0.014598,2.532372,0.121646


In [35]:
enrich_results[enrich_results["FDR_p"]<0.05].sort_values(by=["Pathway","treatment","physiotype","Class"])

,Pathway,Class,treatment,physiotype,p-value,odds_ratio,FDR_p
214,CA-cycle,UP,D,C3+CAM,0.000674,0.135416,0.049837
264,CA-cycle,UP,D,CAM,0.000704,0.086642,0.049837
289,CA-cycle,UP,D,all,0.000055,0.106020,0.016474
89,CA-cycle,UP,all,all,0.001827,0.284591,0.049837
240,CAM,ADD,D,facultative CAM,0.000241,3.633718,0.036155
15,CAM,ADD,all,C3+CAM,0.001394,3.245004,0.049837
90,CAM,ADD,all,all,0.001809,3.161518,0.049837
40,CAM,ADD,all,facultative CAM,0.001680,3.051075,0.049837
276,PhotResp,ELD_P2,D,all,0.001702,3.545804,0.049837
226,PhotResp,ELD_P2,D,facultative CAM,0.001427,3.242289,0.049837


In [38]:
allpath.to_csv("/home/leviathan22/yucca-genomics/gene_annotations/pathways_annotation_withsyntelogs.csv",sep=",",header=True,
              index=False)